In [86]:
import os
import pandas as pd
import numpy as np

In [87]:
import json

def get_results(model_checkpoints, output_dir='../checkpoints/downstream'):
    dataset, first_experiment = model_checkpoints[0].split('/', 1)
    result_dir = f'{output_dir}/{dataset}/results'
    with open(f'{result_dir}/{first_experiment}/config.json', encoding='utf-8') as file:
        config = json.load(file)
    columns = ['Overall', *config['tasks']]
    results = {col: [] for col in columns}
    mean_values = {col: [] for col in columns}
    model_names = []

    for checkpoint in model_checkpoints:
        experiment_name = checkpoint.split('/', 1)[1]
        filename = f'{result_dir}/{experiment_name}/fold_metrics.csv'
        if not os.path.exists(filename):
            continue
        model_names.append(experiment_name)
        records = pd.read_csv(filename)
        df = records.pivot(index=['repeat', 'fold'], columns='task', values='R2')[config['tasks']]
        category_means = []
        for tasks in config['dataset']['categories'].values():
            tasks = [task for task in tasks if task in df.columns]
            if tasks:
                category_means.append(df[tasks].mean(axis=1))
        df['Overall'] = (pd.concat(category_means, axis=1).mean(axis=1)
                         if category_means else df.mean(axis=1))
        score = df[columns].to_numpy()
        mean = np.mean(score, axis=0)
        std = np.std(score, axis=0, ddof=1 if len(score) > 1 else 0)

        for i, col in enumerate(columns):
            results[col].append(f'{mean[i]:.3f}±{std[i]:.3f}')
            mean_values[col].append(mean[i])

    results['Model'] = model_names
    df = pd.DataFrame(results)
    df.set_index('Model', inplace=True)
    df_mean = pd.DataFrame(mean_values)

    def highlight_max(s):
        x = df_mean[s.name]
        is_max = x == x.max()
        return ['font-weight: bold' if v else '' for v in is_max]

    styled_df = df.style.apply(highlight_max, axis=0)
    return styled_df

In [88]:
model_checkpoints = [
    'MTL/polyBERT',
    'MTL/TransPolymer',
    'MTL/PolyCL',
    'MTL/MMPolymer',
    'MTL/PerioGT',
    'MTL/PoCo',
    'MTL/PoCo_concat',
]
get_results(model_checkpoints)

,Overall,Eat,Xc,Egc,Egb,Eea,Ei,nc,eps
Model,,,,,,,,,
polyBERT,0.734±0.057,0.964±0.013,0.310±0.218,0.872±0.008,0.870±0.049,0.905±0.029,0.735±0.081,0.783±0.059,0.655±0.093
TransPolymer,0.741±0.029,0.974±0.007,0.373±0.106,0.883±0.013,0.872±0.034,0.882±0.015,0.742±0.113,0.753±0.067,0.654±0.082
PolyCL,0.738±0.037,0.964±0.015,0.311±0.152,0.866±0.011,0.840±0.074,0.879±0.029,0.778±0.079,0.791±0.053,0.681±0.107
MMPolymer,0.738±0.036,0.959±0.012,0.352±0.150,0.869±0.017,0.880±0.049,0.883±0.045,0.732±0.113,0.769±0.062,0.666±0.066
PerioGT,0.759±0.041,0.959±0.017,0.369±0.155,0.883±0.015,0.845±0.110,0.886±0.033,0.764±0.083,0.813±0.051,0.724±0.050
PoCo,0.772±0.044,0.985±0.006,0.368±0.156,0.888±0.012,0.896±0.039,0.919±0.019,0.783±0.082,0.807±0.068,0.726±0.074
PoCo_concat,0.778±0.035,0.987±0.004,0.399±0.170,0.890±0.013,0.906±0.039,0.908±0.026,0.790±0.075,0.814±0.037,0.720±0.035


In [89]:
model_checkpoints = [
    'PolyOmics/polyBERT',
    'PolyOmics/TransPolymer',
    'PolyOmics/PolyCL',
    'PolyOmics/MMPolymer',
    'PolyOmics/PerioGT',
    'PolyOmics/PoCo',
    'PolyOmics/PoCo_concat',
]
get_results(model_checkpoints)

,Overall,thermal_conductivity,thermal_diffusivity,CLTE,tg,density,Rg,self-diffusion,Cp,Cv,qm_homo_monomer1,qm_lumo_monomer1,qm_dipole_monomer1,qm_polarizability_monomer1,static_dielectric_const,refractive_index,compressibility,isentropic_compressibility,bulk_modulus,isentropic_bulk_modulus
Model,,,,,,,,,,,,,,,,,,,,
polyBERT,0.784±0.001,0.804±0.006,0.732±0.004,0.694±0.011,0.742±0.006,0.967±0.001,0.825±0.005,0.538±0.031,0.890±0.006,0.918±0.002,0.879±0.004,0.923±0.006,0.531±0.007,0.993±0.002,0.716±0.012,0.961±0.002,0.631±0.012,0.656±0.014,0.698±0.003,0.723±0.001
TransPolymer,0.788±0.003,0.811±0.008,0.743±0.006,0.702±0.008,0.753±0.009,0.958±0.008,0.832±0.004,0.540±0.022,0.897±0.006,0.923±0.003,0.886±0.004,0.924±0.009,0.541±0.007,0.992±0.002,0.717±0.032,0.961±0.002,0.640±0.013,0.669±0.007,0.706±0.004,0.727±0.004
PolyCL,0.786±0.002,0.801±0.009,0.724±0.006,0.701±0.009,0.749±0.012,0.976±0.001,0.807±0.003,0.537±0.028,0.899±0.005,0.926±0.001,0.887±0.006,0.923±0.006,0.533±0.006,0.994±0.002,0.729±0.016,0.961±0.001,0.631±0.010,0.658±0.005,0.702±0.003,0.726±0.004
MMPolymer,0.787±0.002,0.804±0.012,0.740±0.007,0.694±0.010,0.753±0.004,0.969±0.003,0.828±0.006,0.540±0.032,0.899±0.004,0.927±0.003,0.880±0.004,0.920±0.008,0.523±0.006,0.993±0.003,0.732±0.017,0.959±0.003,0.641±0.009,0.668±0.008,0.705±0.004,0.714±0.017
PerioGT,0.797±0.001,0.821±0.003,0.749±0.010,0.711±0.009,0.759±0.005,0.980±0.003,0.830±0.007,0.552±0.037,0.904±0.010,0.935±0.001,0.900±0.006,0.935±0.003,0.555±0.005,0.990±0.003,0.720±0.018,0.962±0.002,0.654±0.006,0.682±0.006,0.720±0.005,0.746±0.004
PoCo,0.795±0.001,0.813±0.008,0.739±0.004,0.714±0.006,0.760±0.011,0.984±0.001,0.813±0.004,0.562±0.026,0.909±0.004,0.934±0.001,0.897±0.003,0.935±0.008,0.541±0.007,0.992±0.002,0.721±0.016,0.965±0.002,0.642±0.013,0.674±0.008,0.718±0.004,0.736±0.009
PoCo_concat,0.799±0.001,0.822±0.009,0.751±0.005,0.717±0.012,0.760±0.012,0.986±0.001,0.829±0.006,0.560±0.024,0.909±0.004,0.937±0.002,0.899±0.005,0.938±0.005,0.549±0.005,0.993±0.003,0.735±0.015,0.965±0.001,0.648±0.011,0.677±0.012,0.715±0.010,0.743±0.005


In [90]:
model_checkpoints = [
    'RadonPy/polyBERT',
    'RadonPy/TransPolymer',
    'RadonPy/PolyCL',
    'RadonPy/MMPolymer',
    'RadonPy/PerioGT',
    'RadonPy/PoCo',
    'RadonPy/PoCo_concat',
]
get_results(model_checkpoints)

,Overall,thermal_conductivity,thermal_diffusivity,linear_expansion,volume_expansion,density,Rg,self-diffusion,Cp,Cv,qm_homo_monomer,qm_lumo_monomer,qm_dipole_monomer,qm_polarizability_monomer,static_dielectric_const,refractive_index,compressibility,isentropic_compressibility,bulk_modulus,isentropic_bulk_modulus
Model,,,,,,,,,,,,,,,,,,,,
polyBERT,0.777±0.015,0.805±0.031,0.761±0.069,0.600±0.079,0.605±0.081,0.943±0.025,0.733±0.074,0.550±0.091,0.938±0.022,0.944±0.023,0.879±0.015,0.859±0.078,0.533±0.063,0.987±0.004,0.816±0.049,0.949±0.009,0.594±0.087,0.617±0.079,0.742±0.037,0.738±0.043
TransPolymer,0.755±0.027,0.766±0.057,0.685±0.083,0.591±0.059,0.585±0.067,0.930±0.023,0.582±0.325,0.559±0.070,0.908±0.102,0.909±0.109,0.900±0.015,0.876±0.056,0.498±0.089,0.992±0.002,0.823±0.049,0.952±0.009,0.572±0.076,0.602±0.094,0.690±0.057,0.686±0.056
PolyCL,0.765±0.014,0.747±0.052,0.688±0.090,0.559±0.094,0.546±0.118,0.943±0.020,0.586±0.127,0.526±0.073,0.938±0.031,0.949±0.029,0.895±0.023,0.846±0.077,0.515±0.062,0.990±0.003,0.823±0.053,0.948±0.013,0.633±0.069,0.639±0.067,0.770±0.035,0.774±0.036
MMPolymer,0.765±0.013,0.782±0.044,0.742±0.060,0.574±0.053,0.583±0.051,0.943±0.015,0.709±0.083,0.546±0.077,0.933±0.038,0.938±0.041,0.900±0.019,0.878±0.058,0.456±0.136,0.993±0.003,0.805±0.076,0.947±0.013,0.592±0.084,0.621±0.072,0.719±0.056,0.707±0.057
PerioGT,0.786±0.015,0.812±0.066,0.769±0.085,0.577±0.070,0.598±0.054,0.963±0.021,0.778±0.064,0.536±0.084,0.955±0.016,0.962±0.014,0.906±0.020,0.883±0.067,0.569±0.079,0.977±0.017,0.774±0.045,0.952±0.010,0.621±0.078,0.651±0.067,0.777±0.043,0.783±0.039
PoCo,0.798±0.012,0.828±0.034,0.794±0.059,0.610±0.060,0.611±0.058,0.965±0.016,0.762±0.066,0.574±0.084,0.956±0.019,0.964±0.020,0.907±0.014,0.873±0.058,0.549±0.061,0.989±0.004,0.832±0.030,0.959±0.012,0.621±0.074,0.634±0.067,0.783±0.030,0.795±0.033
PoCo_concat,0.803±0.011,0.837±0.038,0.813±0.054,0.620±0.053,0.624±0.051,0.970±0.014,0.768±0.084,0.575±0.076,0.959±0.019,0.963±0.024,0.917±0.010,0.878±0.062,0.559±0.054,0.992±0.003,0.830±0.075,0.962±0.004,0.620±0.090,0.643±0.081,0.796±0.021,0.797±0.023


In [91]:
model_checkpoints = [
    'OPC/polyBERT',
    'OPC/TransPolymer',
    'OPC/PolyCL',
    'OPC/MMPolymer',
    'OPC/PerioGT',
    'OPC/PoCo',
    'OPC/PoCo_concat',
]
get_results(model_checkpoints)

,Overall,Tg,Tc,FFV,Density,Rg
Model,,,,,,
polyBERT,0.741±0.023,0.571±0.070,0.798±0.038,0.848±0.043,0.814±0.064,0.733±0.055
TransPolymer,0.735±0.027,0.634±0.071,0.794±0.054,0.838±0.048,0.717±0.135,0.711±0.066
PolyCL,0.727±0.028,0.554±0.070,0.786±0.046,0.837±0.067,0.798±0.074,0.714±0.059
MMPolymer,0.724±0.039,0.595±0.108,0.776±0.048,0.810±0.072,0.758±0.092,0.721±0.050
PerioGT,0.674±0.197,0.289±0.840,0.802±0.056,0.845±0.068,0.852±0.062,0.714±0.055
PoCo,0.758±0.030,0.600±0.063,0.803±0.057,0.879±0.040,0.863±0.072,0.705±0.065
PoCo_concat,0.764±0.033,0.611±0.085,0.801±0.057,0.886±0.034,0.854±0.065,0.725±0.060


In [92]:
model_checkpoints = [
    'Gas/polyBERT',
    'Gas/TransPolymer',
    'Gas/PolyCL',
    'Gas/MMPolymer',
    'Gas/PerioGT',
    'Gas/PoCo',
    'Gas/PoCo_concat',
]
get_results(model_checkpoints)

,Overall,p_exp_CH4,p_exp_CO2,p_exp_H2,p_exp_He,p_exp_N2,p_exp_O2,d_exp_CH4,d_exp_CO2,d_exp_H2,d_exp_He,d_exp_N2,d_exp_O2,s_sim_CH4,s_sim_CO2,s_sim_N2,s_sim_O2
Model,,,,,,,,,,,,,,,,,
polyBERT,0.674±0.040,0.838±0.055,0.782±0.064,0.803±0.045,0.799±0.055,0.820±0.046,0.811±0.053,0.685±0.122,0.577±0.180,0.526±0.233,-0.228±0.638,0.655±0.157,0.760±0.060,0.739±0.088,0.764±0.124,0.788±0.080,0.575±0.205
TransPolymer,0.643±0.052,0.821±0.063,0.780±0.058,0.768±0.098,0.778±0.073,0.790±0.052,0.799±0.034,0.596±0.179,0.505±0.150,0.540±0.148,-0.458±0.788,0.659±0.139,0.710±0.103,0.746±0.089,0.727±0.109,0.740±0.103,0.640±0.177
PolyCL,0.692±0.031,0.831±0.053,0.782±0.065,0.809±0.052,0.803±0.076,0.804±0.050,0.834±0.035,0.671±0.164,0.543±0.212,0.645±0.202,-0.090±0.496,0.635±0.153,0.719±0.100,0.776±0.074,0.759±0.089,0.781±0.063,0.665±0.178
MMPolymer,0.643±0.051,0.831±0.041,0.777±0.043,0.794±0.050,0.795±0.051,0.787±0.056,0.798±0.041,0.591±0.220,0.504±0.150,0.514±0.296,-0.353±0.789,0.616±0.195,0.667±0.120,0.742±0.095,0.722±0.134,0.731±0.113,0.640±0.175
PerioGT,0.708±0.031,0.850±0.045,0.804±0.055,0.816±0.036,0.810±0.049,0.819±0.038,0.828±0.033,0.644±0.140,0.516±0.184,0.504±0.193,0.189±0.197,0.673±0.142,0.682±0.138,0.798±0.056,0.763±0.080,0.787±0.060,0.723±0.097
PoCo,0.708±0.030,0.842±0.043,0.788±0.073,0.792±0.043,0.824±0.052,0.804±0.044,0.823±0.037,0.683±0.182,0.607±0.173,0.513±0.189,0.132±0.376,0.683±0.124,0.750±0.067,0.799±0.055,0.809±0.097,0.782±0.076,0.609±0.237
PoCo_concat,0.698±0.039,0.845±0.040,0.803±0.065,0.799±0.044,0.817±0.065,0.808±0.049,0.814±0.040,0.639±0.190,0.572±0.172,0.539±0.148,0.067±0.490,0.656±0.137,0.746±0.071,0.789±0.058,0.810±0.093,0.778±0.097,0.603±0.229


In [93]:
model_checkpoints = [
    'MTL/PoCo',
    'MTL/PoCo-AbsPE',
    'MTL/PoCo-mask',
    'MTL/PoCo-trans',
    'MTL/PoCo-mult',
    'MTL/PoCo-rand',
    'MTL/PoCo-mask-trans',
    'MTL/PoCo-mask-mult',
    'MTL/PoCo-mask-rand',
    'MTL/PoCo-trans-mult',
    'MTL/PoCo-trans-rand',
    'MTL/PoCo-mult-rand',
]
get_results(model_checkpoints, output_dir='../checkpoints/ablation')

,Overall,Eat,Xc,Egc,Egb,Eea,Ei,nc,eps
Model,,,,,,,,,
PoCo,0.772±0.044,0.985±0.006,0.368±0.156,0.888±0.012,0.896±0.039,0.919±0.019,0.783±0.082,0.807±0.068,0.726±0.074
PoCo-AbsPE,0.738±0.038,0.977±0.010,0.235±0.142,0.864±0.010,0.873±0.047,0.901±0.015,0.769±0.089,0.809±0.057,0.705±0.084
PoCo-mask,0.723±0.048,0.971±0.010,0.279±0.191,0.861±0.011,0.873±0.051,0.909±0.027,0.745±0.119,0.773±0.059,0.618±0.147
PoCo-trans,0.733±0.042,0.972±0.013,0.275±0.190,0.860±0.019,0.864±0.040,0.873±0.026,0.756±0.078,0.792±0.034,0.684±0.056
PoCo-mult,0.707±0.057,0.969±0.008,0.278±0.114,0.859±0.012,0.866±0.048,0.872±0.029,0.737±0.125,0.745±0.097,0.581±0.250
PoCo-rand,0.709±0.039,0.972±0.008,0.270±0.171,0.850±0.011,0.855±0.040,0.866±0.034,0.723±0.107,0.749±0.055,0.614±0.128
PoCo-mask-trans,0.750±0.040,0.972±0.014,0.301±0.153,0.873±0.014,0.880±0.046,0.907±0.023,0.755±0.101,0.823±0.041,0.696±0.098
PoCo-mask-mult,0.736±0.046,0.974±0.010,0.290±0.215,0.872±0.010,0.875±0.057,0.905±0.021,0.751±0.102,0.779±0.039,0.674±0.067
PoCo-mask-rand,0.748±0.037,0.978±0.007,0.317±0.186,0.872±0.011,0.860±0.065,0.900±0.023,0.756±0.080,0.811±0.033,0.686±0.070


In [94]:
model_checkpoints = [
    'MTL/PoCo',
    'MTL/PoCo-H384',
    'MTL/PoCo-H768',
    'MTL/PoCo-L6',
    'MTL/PoCo-L12',
    'MTL/PoCo-Q64',
    'MTL/PoCo-Q256',
    'MTL/PoCo-Q1024',
    'MTL/PoCo-Q4096',
    'MTL/PoCo-Q16k',
    'MTL/PoCo-0.05-0.05',
    'MTL/PoCo-0.07-0.07',
    'MTL/PoCo-0.07-0.05',
    'MTL/PoCo-0.03-0.07',
    'MTL/PoCo-0.05-0.10',
]
get_results(model_checkpoints, output_dir='../checkpoints/ablation')

,Overall,Eat,Xc,Egc,Egb,Eea,Ei,nc,eps
Model,,,,,,,,,
PoCo,0.772±0.044,0.985±0.006,0.368±0.156,0.888±0.012,0.896±0.039,0.919±0.019,0.783±0.082,0.807±0.068,0.726±0.074
PoCo-H384,0.772±0.032,0.987±0.005,0.323±0.153,0.884±0.014,0.901±0.036,0.919±0.021,0.770±0.091,0.837±0.035,0.747±0.047
PoCo-H768,0.773±0.028,0.983±0.009,0.357±0.146,0.889±0.010,0.892±0.040,0.915±0.018,0.778±0.091,0.820±0.038,0.742±0.061
PoCo-L6,0.766±0.036,0.978±0.005,0.338±0.152,0.888±0.008,0.903±0.038,0.922±0.019,0.782±0.087,0.815±0.045,0.716±0.068
PoCo-L12,0.760±0.042,0.976±0.010,0.321±0.172,0.885±0.013,0.900±0.047,0.920±0.018,0.787±0.081,0.800±0.064,0.716±0.072
PoCo-Q64,0.747±0.036,0.979±0.007,0.303±0.163,0.854±0.014,0.857±0.061,0.889±0.014,0.770±0.076,0.808±0.051,0.706±0.075
PoCo-Q256,0.757±0.035,0.984±0.005,0.306±0.138,0.869±0.013,0.875±0.040,0.895±0.019,0.772±0.082,0.824±0.044,0.725±0.072
PoCo-Q1024,0.764±0.042,0.984±0.006,0.326±0.180,0.879±0.011,0.891±0.051,0.899±0.021,0.783±0.077,0.828±0.042,0.723±0.067
PoCo-Q4096,0.772±0.035,0.987±0.004,0.339±0.176,0.882±0.010,0.898±0.039,0.903±0.024,0.788±0.077,0.835±0.036,0.735±0.055
